In [1]:
# analysis
import numpy as np
import pandas as pd 

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell
import scanpy as sc
import mudata as mu
import liana as li

# import ginsim
import maboss

from pathlib import Path
import os 

# seting global dir
cwd=Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent) 
os.getcwd()

ipylab module is not installed, menus and toolbar are disabled.


'/home/maxi7524/repositories/MSB'

In [8]:
# load model zginml format
maboss_file = Path('data/pymyboss/CellFateModel')
master_model = maboss.load(maboss_file.with_suffix('.bnd')._str, maboss_file.with_suffix('.cfg')._str)
master_model.network

Network([('FASL', <maboss.network.Node at 0x7dcb9fe69a30>),
         ('TNF', <maboss.network.Node at 0x7dcb9ffd9100>),
         ('TNFR', <maboss.network.Node at 0x7dcb9ffd8d10>),
         ('DISC_TNF', <maboss.network.Node at 0x7dcb9ffd98e0>),
         ('DISC_FAS', <maboss.network.Node at 0x7dcb9ffdb530>),
         ('FADD', <maboss.network.Node at 0x7dcb9ffdaf90>),
         ('RIP1', <maboss.network.Node at 0x7dcb9ffdb1a0>),
         ('RIP1ub', <maboss.network.Node at 0x7dcb9ffd9d90>),
         ('RIP1K', <maboss.network.Node at 0x7dcb9ffdb710>),
         ('IKK', <maboss.network.Node at 0x7dcb9ffda990>),
         ('NFkB', <maboss.network.Node at 0x7dcb9ffdbce0>),
         ('CASP8', <maboss.network.Node at 0x7dcb9ffd9f70>),
         ('BAX', <maboss.network.Node at 0x7dcb9ffda3f0>),
         ('BCL2', <maboss.network.Node at 0x7dcb9ffd8950>),
         ('mROS', <maboss.network.Node at 0x7dcb9ffd8ef0>),
         ('ROS', <maboss.network.Node at 0x7dcb9ffdb140>),
         ('ATP', <maboss.network

In [9]:
# here we add value like from liana data
target_receptor = "FASL" # first from list 
communication_score = 0.85

master_model.network.set_istate(target_receptor, [1 - communication_score, communication_score])

In [10]:
# run simulation
results = master_model.run()

In [11]:
def map_liana_to_maboss(master_model, liana_res, target_cell_type, spatial=False):
    """
    Integrates LIANA+ consensus results into a MaBoSS model.
    
    Args:
        master_model: The loaded pyMaBoSS model object.
        liana_res (pd.DataFrame): The result table from LIANA+.
        target_cell_type (str): The specific cell type/cluster we are modeling as the 'target'.
        spatial (bool): If True, prepares data for spatial agent-based initialization (PhysiBoSS).
        
    Returns:
        Modified master_model or a configuration dict for PhysiBoSS.
    """
    
    # 1. Filter LIANA results for our target cell type
    # We focus on interactions where our modeled cell is the 'receiver' (target)
    relevant_interactions = liana_res[liana_res['target'] == target_cell_type].copy()
    
    if relevant_interactions.empty:
        print(f"Warning: No interactions found for target {target_cell_type}")
        return master_model

    # 2. Normalize the score (assuming LIANA consensus score or magnitude)
    # We need a value between 0 and 1 for MaBoSS probabilities
    # Here we assume 'magnitude' column exists; adjust based on your LIANA+ output
    score_col = 'magnitude' if 'magnitude' in relevant_interactions.columns else 'aggregate_rank'
    
    # 3. Map receptors to model nodes
    # Note: You might need a dictionary if model node names differ from LIANA receptor names
    model_nodes = list(master_model.network.keys())
    
    for index, row in relevant_interactions.iterrows():
        receptor = row['receptor']
        score = row[score_col]
        
        # Check if the receptor from LIANA exists as a node in our Boolean network
        if receptor in model_nodes:
            if not spatial:
                # NON-SPATIAL CASE (RNA/CITE-seq)
                # Set global Initial State (istate) based on average interaction strength
                # [Prob(0), Prob(1)]
                master_model.network.set_istate(receptor, [1 - score, score])
                print(f"Global: Set {receptor} initial state to {score}")
            else:
                # SPATIAL CASE (PhysiBoSS)
                # In PhysiBoSS, we often want to modify transition rates ($u_Node) 
                # to represent how fast a receptor activates in a specific spatial context.
                # Here we modify the 'up' rate parameter in the configuration.
                master_model.param[f"${receptor}_up"] = score
                print(f"Spatial: Set {receptor} activation rate ($up) to {score}")

    return master_model